# 12yr measurement-model fitting: NoInhibition and RecallOverride

For each model, the free parameters are:

- $p_{\text{intention}}$
- $p_{\text{recall}}$

1. load the retained participant-level fitting sample,
2. define the two measurement-model prediction functions,
3. fit each participant by maximum likelihood under a binomial likelihood,
4. summarize fitted parameter distributions,
5. compare observed vs predicted AX / AY / BX / BY accuracies,
6. save the fitted outputs for later group analyses.

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import binom

In [3]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

fit_path = project_root / "data" / "AXCPT_12yr_fitting_sample.csv"

print("Project root:", project_root)
print("Fitting file:", fit_path)
print("Exists:", fit_path.exists())

fit_df = pd.read_csv(fit_path)

print("Loaded:", fit_path)
print("Shape:", fit_df.shape)

Project root: /Users/yizj/Desktop/AX-CPT/metaControl_behavioral_Inhibition
Fitting file: /Users/yizj/Desktop/AX-CPT/metaControl_behavioral_Inhibition/data/AXCPT_12yr_fitting_sample.csv
Exists: True
Loaded: /Users/yizj/Desktop/AX-CPT/metaControl_behavioral_Inhibition/data/AXCPT_12yr_fitting_sample.csv
Shape: (129, 41)


### Define trial-type columns and fixed settings

In [4]:
TRIAL_TYPES = ["AX", "AY", "BX", "BY"]

count_cols = [
    "AX_Count_Filtered_12yr",
    "AY_Count_Filtered_12yr",
    "BX_Count_Filtered_12yr",
    "BY_Count_Filtered_12yr",
]

acc_count_cols = [
    "AX_Acc_Count_12yr",
    "AY_Acc_Count_12yr",
    "BX_Acc_Count_12yr",
    "BY_Acc_Count_12yr",
]

acc_rate_cols = [
    "AX_Acc_12yr",
    "AY_Acc_12yr",
    "BX_Acc_12yr",
    "BY_Acc_12yr",
]

p_cue_probe = np.array([0.70, 0.10, 0.10, 0.10], dtype=float)

load = 1
m = 1.0
p_slip = 1 - fit_df["BY_Acc_12yr"].mean()
lambda_ = 0.0

print("Using p_cue_probe =", p_cue_probe)
print("Using load =", load)
print("Using m =", m)
print("Using p_slip =", p_slip)
print("Using lambda_ =", lambda_)

Using p_cue_probe = [0.7 0.1 0.1 0.1]
Using load = 1
Using m = 1.0
Using p_slip = 0.15375528863900956
Using lambda_ = 0.0


### Component accuracies

Both measurement models use the same component-level accuracies:

- habitual accuracy,
- proactive accuracy,
- recall accuracy.

These are treated as model constants.

In [5]:
def compute_component_accuracies(p_cue_probe, load=1, m=1.0, p_slip=0.0, lambda_=0.0):
    """
    Returns habit, proactive, and recall accuracies for [AX, AY, BX, BY].
    """
    pAX, pAY, pBX, pBY = p_cue_probe

    p_correct_habit = np.array([
        pAX / (pAX + pBX),   # AX
        1.0,                 # AY
        pBX / (pAX + pBX),   # BX
        1.0                  # BY
    ], dtype=float)

    proactive_base = np.array([
        pAX / (pAX + pAY),         # AX
        1.0 - pAX / (pAX + pAY),   # AY
        1.0,                       # BX
        1.0                        # BY
    ], dtype=float)
    proactive_base = proactive_base * (1.0 - lambda_ * (load - 1))

    recall_base = np.full(4, m * (1.0 - lambda_ * (load - 1)), dtype=float)

    def apply_slip(p):
        return p * (1 - p_slip) + (1 - p) * p_slip

    p_correct_habit = apply_slip(p_correct_habit)
    p_correct_proactive = apply_slip(proactive_base)
    p_correct_recall = apply_slip(recall_base)

    return p_correct_habit, p_correct_proactive, p_correct_recall

## NoInhibition measurement model

Under NoInhibition,

$$
P(\text{correct})
=
p_{\text{intention}} \, Acc_{\text{proactive}}
+
(1-p_{\text{intention}})
\Big[
p_{\text{recall}} \, Acc_{\text{recall}}
+
(1-p_{\text{recall}})\, Acc_{\text{habitual}}
\Big].
$$

In [11]:
def predict_accuracies_noinhibition_2param(params, p_cue_probe, load=1, m=1.0, p_slip=0.0, lambda_=0.0):
    p_intention, p_recall = params

    p_habit, p_pro, p_rec = compute_component_accuracies(
        p_cue_probe=p_cue_probe,
        load=load,
        m=m,
        p_slip=p_slip,
        lambda_=lambda_
    )

    preds = np.zeros(4)
    for s in range(4):
        preds[s] = (
            p_intention * p_pro[s]
            + (1 - p_intention) * (
                p_recall * p_rec[s]
                + (1 - p_recall) * p_habit[s]
            )
        )

    return np.clip(preds, 1e-8, 1 - 1e-8)

## RecallOverride measurement model

Under RecallOverride,

$$
P(\text{correct})
=
p_{\text{recall}} \, Acc_{\text{recall}}
+
(1-p_{\text{recall}})
\Big[
p_{\text{intention}} \, Acc_{\text{proactive}}
+
(1-p_{\text{intention}})\, Acc_{\text{habitual}}
\Big].
$$

In [6]:
def predict_accuracies_recalloverride_2param(params, p_cue_probe, load=1, m=1.0, p_slip=0.0, lambda_=0.0):
    p_intention, p_recall = params

    p_habit, p_pro, p_rec = compute_component_accuracies(
        p_cue_probe=p_cue_probe,
        load=load,
        m=m,
        p_slip=p_slip,
        lambda_=lambda_
    )

    preds = np.zeros(4)
    for s in range(4):
        preds[s] = (
            p_recall * p_rec[s]
            + (1 - p_recall) * (
                p_intention * p_pro[s]
                + (1 - p_intention) * p_habit[s]
            )
        )

    return np.clip(preds, 1e-8, 1 - 1e-8)

## Define the binomial negative log-likelihood

For each participant, the observed data are:

- 4 filtered trial counts,
- 4 correct counts.

The participant likelihood is the product of 4 binomial terms.

In [7]:
def neg_loglik_model(params, n_trials, n_correct, predict_fn, p_cue_probe, load=1, m=1.0, p_slip=0.0, lambda_=0.0):
    preds = predict_fn(
        params,
        p_cue_probe=p_cue_probe,
        load=load,
        m=m,
        p_slip=p_slip,
        lambda_=lambda_
    )
    ll = binom.logpmf(n_correct.astype(int), n_trials.astype(int), preds).sum()
    return -ll

## Fit both measurement models

In [ ]:
def fit_one_participant(
    n_trials,
    n_correct,
    predict_fn,
    p_cue_probe,
    load=1,
    m=1.0,
    p_slip=0.0,
    lambda_=0.0,
    n_starts=30,
    seed=123
):
    rng = np.random.default_rng(seed)
    bounds = [(0.0, 1.0), (0.0, 1.0)]

    best_res = None

    for _ in range(n_starts):
        x0 = rng.uniform(0.05, 0.95, size=2)

        res = minimize(
            neg_loglik_model,
            x0=x0,
            args=(n_trials, n_correct, predict_fn, p_cue_probe, load, m, p_slip, lambda_),
            bounds=bounds,
            method="L-BFGS-B"
        )

        if best_res is None or res.fun < best_res.fun:
            best_res = res

    est = best_res.x
    preds = predict_fn(
        est,
        p_cue_probe=p_cue_probe,
        load=load,
        m=m,
        p_slip=p_slip,
        lambda_=lambda_
    )

    return {
        "success": best_res.success,
        "fun": best_res.fun,
        "params": est,
        "preds": preds,
        "message": best_res.message
    }

In [9]:
def fit_real_sample(fit_df, predict_fn, model_name, count_cols, acc_count_cols, p_cue_probe, load=1, m=1.0, p_slip=0.0, lambda_=0.0):
    rows = []

    for _, row in fit_df.iterrows():
        n_trials = row[count_cols].to_numpy(dtype=int)
        n_correct = row[acc_count_cols].to_numpy(dtype=int)

        fit_res = fit_one_participant(
            n_trials=n_trials,
            n_correct=n_correct,
            predict_fn=predict_fn,
            p_cue_probe=p_cue_probe,
            load=load,
            m=m,
            p_slip=p_slip,
            lambda_=lambda_,
            n_starts=30,
            seed=int(row["ID"])
        )

        rows.append({
            "ID": row["ID"],
            "model": model_name,
            "fit_success": fit_res["success"],
            "nll": fit_res["fun"],
            "p_intention": fit_res["params"][0],
            "p_recall": fit_res["params"][1],
            "pred_AX": fit_res["preds"][0],
            "pred_AY": fit_res["preds"][1],
            "pred_BX": fit_res["preds"][2],
            "pred_BY": fit_res["preds"][3],
            "obs_AX": row["AX_Acc_12yr"],
            "obs_AY": row["AY_Acc_12yr"],
            "obs_BX": row["BX_Acc_12yr"],
            "obs_BY": row["BY_Acc_12yr"],
        })

    return pd.DataFrame(rows)

In [ ]:
fit_NI = fit_real_sample(
    fit_df=fit_df,
    predict_fn=predict_accuracies_noinhibition_2param,
    model_name="NoInhibition",
    count_cols=count_cols,
    acc_count_cols=acc_count_cols,
    p_cue_probe=p_cue_probe,
    load=load,
    m=m,
    p_slip=p_slip,
    lambda_=lambda_
)

fit_RO = fit_real_sample(
    fit_df=fit_df,
    predict_fn=predict_accuracies_recalloverride_2param,
    model_name="RecallOverride",
    count_cols=count_cols,
    acc_count_cols=acc_count_cols,
    p_cue_probe=p_cue_probe,
    load=load,
    m=m,
    p_slip=p_slip,
    lambda_=lambda_
)

print(fit_NI.shape, fit_RO.shape)